#### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import pickle
import warnings
import os
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')

from kmrf import KMRF
from KMRF_training_config import *
from MODEL_INFO import MODEL_INFO
from ANALYTICAL_INPUTS import ANALYTICAL_INPUTS
from PORTFOLIO_OPTIMIZER import PORTFOLIO_OPTIMIZER
from BACKTEST_ANALYTICAL import BACKTEST

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


#### Global Vars and Helper Functions

In [2]:
KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')
KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')

# Using saved KAMA+MSR models
def get_KM_model_dates(KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) -> pd.Series:
    return pd.Series([f.stem for f in list(KM_MODEL_BASE_PATH.glob('*'))]).sort_values().iloc[1:].reset_index(drop=True)

def get_KM_model_paths(MODEL_DATE: str, KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) ->  pd.Series:
    return pd.Series(list((KM_MODEL_BASE_PATH / MODEL_DATE).glob('*'))).sort_values().reset_index(drop=True)

# Using saved KMRF predictions
def get_asset_names(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    kmrf_preds_paths = list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))
    return pd.Series([f.stem.split('multi')[0][:-1].replace('_', ' ') for f in kmrf_preds_paths]).sort_values().reset_index(drop=True)

def get_KMRF_prediction_paths(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    return pd.Series(list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))).sort_values().reset_index(drop=True)

In [3]:
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

TRADING_DAYS = CustomBusinessDay(calendar=USFederalHolidayCalendar())

RF_RATES_TS = pd.read_csv('data/risk_free_rates.csv', parse_dates=['Date'], index_col='Date')[['RF_3M']].sort_index()

# -----------------------------

In [4]:
get_asset_names()

0     Consumer Discretionary Select Sector SPDR
1           Consumer Staples Select Sector SPDR
2                     Energy Select Sector SPDR
3                  Financial Select Sector SPDR
4                Health Care Select Sector SPDR
5                 Industrial Select Sector SPDR
6                        Invesco DB Agriculture
7                             Invesco QQQ Trust
8                  Materials Select Sector SPDR
9         SPDR Dow Jones Industrial Average ETF
10                             SPDR Gold Shares
11                             SPDR S&P 500 ETF
12                Technology Select Sector SPDR
13               United States Natural Gas Fund
14                       United States Oil Fund
15                 Utilities Select Sector SPDR
16          Vanguard FTSE Developed Markets ETF
17           Vanguard FTSE Emerging Markets ETF
18                     Vanguard FTSE Europe ETF
19                  iShares China Large-Cap ETF
20                       iShares MSCI In

In [5]:
KM_MODEL_DATES = get_KM_model_dates()
KM_MODEL_DATES.iloc[-1]

'20251007'

In [6]:
PORTFOLIO_ASSETS = ['SPDR S&P 500 ETF', 'iShares Russell 2000 ETF',
                    'iShares Micro-Cap ETF',
                    'SPDR Dow Jones Industrial Average ETF',
                    'iShares China Large-Cap ETF',
                    'Vanguard FTSE Emerging Markets ETF',
                    'iShares U.S. Real Estate ETF', 
                    'SPDR Gold Shares', 'iShares Silver Trust']

In [ ]:
bt = BACKTEST(asset_list = PORTFOLIO_ASSETS,
        start_date=KM_MODEL_DATES.iloc[-46],
        end_date= '20251031',
        rebalance_frequency= 10,
        objective= 'max_sharpe',
        allow_short_selling= True,
        gross_exposure_limit= 2.0,
        min_weight= -0.2,
        max_weight= 0.4,
        max_turnover= 0.25,
        risk_aversion= 1.0,
        n_days= 10,
        initial_capital= 100_000.0,
        transaction_cost_bps= 5.0)

In [ ]:
bt.run_parallel(n_jobs=4)


RUNNING BACKTEST (PARALLEL)
  Period: 2021-12-30 to 2025-10-31
  Assets: 9
  Rebalance frequency: Every 10 trading days
  Rebalance dates: 97
  Objective: max_sharpe
  Short selling: Allowed
  Gross exposure limit: 200.0%
  Parallel jobs: 4

  Computing optimizations in parallel...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
